# Exploring the Titanic: A Narrated Session

DS 2023 | Communicating with Data

## Overview

This notebook is a record of an exploration, written as it happens.

We start with a question, run a little code, look at the result, and let what we see shape the next question.

Along the way we name what we are doing with the vocabulary of *Gotz and Zhou's Task Framework*.

Our **task** is the big question we are trying to answer.

The **sub-tasks** are the smaller questions we break it into.

The **actions** are the steps we take in code: we *Query* the data, *Inspect* it, *Split* it by a group, *Filter* it, *Sort* it, or *Change* the *Metaphor* by trying a different kind of plot.

When we write down something we learned, that is an **insight**, and it closes a **trail**.

And often an insight raises a new question, which becomes the next sub-task, or a new task altogether.

Nothing here is planned in advance.

That is the point.

```{note}
Action names from Gotz and Zhou appear in *italics*, such as *Split* and *Inspect*.
They describe what we are doing, so that we can look back later and see how each finding was reached.
```

## Set Up

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Who survived?

Here is our question: *who survived the sinking of the Titanic?*

That is our task.

It is far too vague to answer directly.

We don't yet know what the data say about a person, so we can't say what *who* means.

So our first sub-task is simply to see what we have.

Let's *Query* the data and look at the first few rows.

In [ ]:
titanic = sns.load_dataset('titanic')
titanic.head()

Each row is a passenger.

`survived` is 1 or 0, which is what we care about.

Several columns describe the passenger: `sex`, `age`, `class`, where they boarded, whether they traveled alone.

But something looks odd.

`pclass` and `class` both seem to hold the ticket class, and `survived` and `alive` seem to say the same thing.

Before we use any of these, we should know which ones are real.

## Are some columns saying the same thing twice?

This is a new sub-task, raised by what we just saw.

Let's *Inspect* `class` against `pclass`.

In [ ]:
pd.crosstab(titanic['class'], titanic.pclass)

Every *First* is a 1, every *Second* a 2, every *Third* a 3.

They are the same column in two forms.

Let's check the other pair we noticed.

In [ ]:
pd.crosstab(titanic.alive, titanic.survived)

The same again.

And by the look of the names, `embarked` and `embark_town` are a third pair.

In [ ]:
pd.crosstab(titanic.embark_town, titanic.embarked)

So we can set aside `pclass`, `alive`, and `embarked`.

That is our first insight, and it closes this short trail: three **redundant** columns, in the language of *Pandas Basics*.

It is not a finding about the Titanic.

It is a finding about the data, and it tells us which columns to use from here on.

## How many survived?

Back to the main task.

The simplest version of *who survived* is *how many*.

Let's *Inspect* the outcome.

In [ ]:
titanic.survived.value_counts()

In [ ]:
titanic.survived.mean()

342 of 891 passengers survived, about 38%.

So fewer than four in ten.

That is a number to remember, because every comparison we make will be against it.

Now we can make the question more specific: *were some kinds of passengers more likely to survive than others?*

That is a refinement of the task, and it suggests a sub-task for each column that describes a passenger.

The obvious place to start is sex.

## Did women survive more often than men?

Let's *Split* the survival rate by sex.

In [ ]:
by_sex = titanic.groupby('sex').survived.mean()
by_sex

In [ ]:
ax = by_sex.plot.barh(figsize=(6, 1.8), xlim=(0, 1), color='tab:blue', title='Share who survived, by sex')
ax.set_ylabel('')
plt.show()

That is a huge difference.

About three women in four survived, and fewer than one man in five.

We'll write that down as an insight.

But a gap this large makes us suspicious.

Could it be hiding something else?

Maybe men were more likely to be in third class, and third class did worse.

That is a new question, and to answer it we need to know about class first.

## Did class matter?

Let's *Split* by class.

In [ ]:
by_class = titanic.groupby('class', observed=True).survived.mean()
by_class

Survival falls steadily with class: about 63% in first, 47% in second, 24% in third.

Another insight.

Now we can return to the suspicion we had about sex.

## Were men more likely to be in third class?

This sub-task was raised by our first insight.

Let's *Inspect* how the sexes were spread across the classes.

In [ ]:
pd.crosstab(titanic['class'], titanic.sex, margins=True)

347 of the 577 men were in third class, about 60%.

For women it was 144 of 314, about 46%.

So yes, men were more concentrated in third class.

But that alone can't explain a gap of 74% against 19%.

The real test is to compare women and men *within* each class.

That means a *Split* by two variables at once.

In [ ]:
by_class_sex = titanic.pivot_table(index='class', columns='sex', values='survived',
                                   aggfunc='mean', observed=True)
by_class_sex.round(2)

In [ ]:
ax = by_class_sex.plot.bar(rot=0, figsize=(6, 3), ylim=(0, 1), color=['tab:orange', 'tab:blue'],
                           title='Share who survived, by class and sex')
ax.set_xlabel('')
plt.show()

In every class, women survived far more often than men.

And for both sexes, survival falls with class.

So the sex gap is not hiding a class gap: both are real, and they add up.

Let's *Sort* the six groups to see the extremes.

In [ ]:
rates = by_class_sex.stack().sort_values()
rates.round(2)

97% of first-class women survived.

14% of third-class men did.

This is the strongest finding so far, and it closes the trail that began with our suspicion about sex.

Let's keep this view, so we can come back to it.

In Gotz and Zhou's terms, saving a view is a *Bookmark*; in a notebook, it is a variable.

In [ ]:
six_groups = rates.copy()

## Did it matter where passengers boarded?

Sex and class were the obvious places to look.

Now a less obvious one: the port where each passenger boarded.

Let's *Split* by port.

In [ ]:
by_port = titanic.groupby('embark_town').survived.mean().sort_values()
by_port

Passengers who boarded at Cherbourg survived at 55%, much more than the 34% from Southampton.

That is surprising.

Why would the port matter?

A ship doesn't care where you got on.

One possibility comes straight from what we just learned: maybe Cherbourg served more first-class passengers.

That's our next sub-task.

Let's *Inspect* the class mix at each port.

In [ ]:
pd.crosstab(titanic.embark_town, titanic['class'], normalize='index').round(2)

About half of the Cherbourg passengers were in first class, compared with a fifth at Southampton.

So part of the port gap really is the class gap.

But is that all of it?

Let's *Split* by port and class together.

In [ ]:
titanic.pivot_table(index='embark_town', columns='class', values='survived',
                    aggfunc='mean', observed=True).round(2)

Within every class, Cherbourg passengers still did better than Southampton passengers.

In third class the difference is large, 38% against 19%.

So class explains some of the port gap, but not all of it, and nothing we have looked at explains the rest.

We could keep going: perhaps the sex mix was different at each port.

But this is a side road.

It doesn't change the main story, so we note it as an open question and turn back.

That, too, is an insight: knowing what we don't know, and why we stopped.

## Women and children first?

Everyone has heard the phrase.

Did children actually survive more often?

The `who` column looks like it divides passengers into men, women, and children.

Let's *Split* by it.

In [ ]:
titanic.groupby('who').survived.mean().sort_values()

Children survived at 59%, between men at 16% and women at 76%.

But how does this dataset know who is a child?

It must use age.

And that makes us want to *Inspect* age itself before trusting the result.

In [ ]:
titanic.age.describe()

The count is 714, not 891.

So age is missing for 177 passengers, about one in five.

That is a lot.

Let's see what happens if we try the obvious thing and *Filter* to children by age.

In [ ]:
children = titanic.query('age < 16')
len(children), children.survived.mean().round(2)

That gives a sensible-looking answer.

But notice what the filter silently did: every passenger with a missing age was dropped, because a missing age is never less than 16.

So the question becomes: are the missing ages just random gaps, or are they a particular kind of passenger?

Let's take the filter back, which Gotz and Zhou would call an *Undo*, and *Split* the missing ages by class instead.

In [ ]:
titanic.age.isna().groupby(titanic['class'], observed=True).mean().round(2)

More than a quarter of third-class ages are missing, against one in seven in first class and one in sixteen in second.

So the missing ages are not random.

Did those passengers survive at a different rate?

In [ ]:
titanic.groupby(titanic.age.isna()).survived.mean().round(2)

Passengers with a known age survived at 41%.

Passengers with a missing age survived at 29%.

So any finding that uses age is drawn from a group that leans toward the upper classes, and toward survivors.

That doesn't mean children didn't do better.

It means we should be careful about saying how much better.

Let's *Change* the *Metaphor* and look at the whole distribution of known ages, for survivors and for those who died.

In [ ]:
ages = (titanic
        .pivot(columns='survived', values='age')
        .rename(columns={0: 'died', 1: 'survived'})
        .rename_axis(columns=None))

ax = ages.plot.hist(bins=range(0, 85, 5), alpha=0.6, color=['tab:gray', 'tab:blue'],
                    figsize=(6, 3), title='Age by outcome, known ages only')
ax.set_xlabel('Age (years)')
plt.show()

Among children under five, survivors outnumber those who died by about two to one.

From five to ten the two are even, and past ten the pattern reverses and stays that way.

So *children first* shows up in the data, but mostly for young children, and with the caution about missing ages attached.

We'll note it, with the caution.

## Looking back

Let's stop and look at where we've been.

We started with one vague task, *who survived?*, and it became a chain of smaller questions, each raised by the answer to the one before.

- Looking at the columns raised a question about **redundant columns**, which we settled first.
- *How many survived?* gave us the baseline: 38%.
- *Did sex matter?* found a huge gap, which raised a suspicion about **class**.
- *Did class matter?* found a second gap, and splitting by **both** showed that the two add up, from 97% of first-class women to 14% of third-class men.
- *Did the port matter?* looked surprising, turned out to be partly class, and left an **open question** we chose not to chase.
- *Women and children first?* led us to **age**, and age led us to the problem of missing values, which tempered the finding.

Some of these trails ended in findings about the Titanic.

Some ended in findings about the data.

One ended in a question we left open.

All of them are worth keeping, because together they explain why we believe what we believe.

```{important}
We looked at several variables before settling on a story.
The more ways we slice the data, the more likely one slice looks interesting by chance: Gelman and Loken's **garden of forking paths**.
The sex and class findings are very large and match the historical record, so they are safe.
The port finding is exactly the kind that could be a fluke, which is one more reason not to build on it.
```

## Where this could go next

An exploration is not something we hand to a reader.

But it is what we build from.

If we were making an **infographic**, we would take the strongest trail, sex and class, and present its findings: 38% overall, 74% against 19%, survival falling with class, and the six groups sorted from 97% to 14%.

We would leave the port out, and leave children out or mention them with care.

That is the infographic in *Infographic Design*.

If we were making a **dashboard**, we would look at what we kept *doing*.

Over and over, we *Split* the survival rate by one variable, and then by two.

Those two moves are what a reader would want to repeat for themselves, so they become two dropdowns: *group by*, and *then split by*.

That is the kind of dashboard built in *Dashboards*.

Either way, the exploration comes first, and the artifact is built from what it found.

## Questions

1. Where in this session did one finding raise the next question? Find three places.
2. We stopped chasing the port. Write the next question you would ask, and the code you would run to answer it.
3. Try `titanic.groupby('alone').survived.mean()`. What did traveling alone have to do with survival? What would you check next?
4. We used `who` to find children. Using what you know about missing ages, write a sentence about children's survival that you would be willing to defend.
5. Pick another dataset from the course, such as `penguins` or `tips`. Start with one vague question, and write your own narrated session of at least five steps.

## References

- Gotz, D., and M. X. Zhou. 2009. "Characterizing Users' Visual Analytic Activity for Insight Provenance." *Information Visualization* 8 (1): 42–55.
- Tukey, J. W. 1977. *Exploratory Data Analysis*. Addison-Wesley.
- Gelman, A., and E. Loken. 2014. "The Statistical Crisis in Science." *American Scientist* 102 (6): 460–465.
- See also, in this book: *Gotz and Zhou's Task Framework*, *Infographic Design*, and *Dashboards*.